# Experiment 1: is the 10% no-trade band worth it?

Strategy: the configured DCC-GARCH model with the 60-month calibration rescale (`config/risk_config.json`), primary cost 2 bps, 1976-2014, data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`). Earlier reports under `reports/*_20261026/` used the rolling-covariance model and are left untouched. Logic lives in `scripts/run_band_experiment.py`; this notebook runs it and reads the output in `reports/dcc_garch_studies/band/`.

## Goal

Compare bands of 0%, 5% and 10% (reference) at 1 and 2 bps. Decision rule, fixed in `config/band_experiment.json` before the 5% cases were computed: a TREND candidate needs at least 10 bps/yr gain at both costs, positive Bonferroni-adjusted paired-bootstrap lower bounds for all three block lengths, and gains in at least 3 of 4 subperiods at both costs. No automatic promotion.

## Run

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
REFERENCE = ROOT / "reports/cost_revision_20261026"  # hashed Step 2 inputs (data only)
STUDIES = ROOT / "reports/dcc_garch_studies"
pd.options.display.float_format = "{:.4f}".format
from run_band_experiment import main

OUT = STUDIES / "band"
if not OUT.exists():
    main(REFERENCE, OUT)

## Performance by band

In [2]:
comparison = pd.read_csv(OUT / "comparison.csv")
comparison[["band", "cost_bps", "strategy", "cagr", "sharpe", "max_drawdown", "turnover_ann", "limit_override_months"]]

,band,cost_bps,strategy,cagr,sharpe,max_drawdown,turnover_ann,limit_override_months
0,0.0000,1,trend,0.1160,1.2191,-0.1066,10.4887,428
1,0.0000,1,static,0.0463,0.5032,-0.4925,3.4828,467
2,0.0000,2,trend,0.1149,1.2080,-0.1068,10.4882,428
3,0.0000,2,static,0.0460,0.4997,-0.4931,3.4827,467
4,0.0500,1,trend,0.1144,1.2131,-0.1114,9.3809,146
5,0.0500,1,static,0.0465,0.5103,-0.4842,2.4787,131
6,0.0500,2,trend,0.1133,1.2032,-0.1115,9.3813,146
7,0.0500,2,static,0.0463,0.5078,-0.4847,2.4788,131
8,0.1000,1,trend,0.1128,1.2083,-0.1162,8.6696,122
9,0.1000,1,static,0.0460,0.5111,-0.4756,1.8695,108


## Paired difference vs the 10% band

`gross_difference_ann` is the return from holding closer to target; `cost_savings_ann` is the fee saved (negative means the narrower band cost more).

In [3]:
pairs = pd.read_csv(OUT / "paired_comparison.csv")
pairs.drop(columns="run")

,band,cost_bps,strategy,gross_difference_ann,cost_savings_ann,net_difference_ann
0,0.0000,1,trend,0.0033,-0.0002,0.0031
1,0.0000,1,static,0.0008,-0.0002,0.0006
2,0.0000,2,trend,0.0033,-0.0004,0.0029
3,0.0000,2,static,0.0008,-0.0003,0.0005
4,0.0500,1,trend,0.0016,-0.0001,0.0015
5,0.0500,1,static,0.0008,-0.0001,0.0007
6,0.0500,2,trend,0.0016,-0.0001,0.0014
7,0.0500,2,static,0.0008,-0.0001,0.0007


## Bootstrap intervals, subperiods and decision rule

In [4]:
intervals = pd.read_csv(OUT / "paired_intervals.csv")
trend = intervals[intervals.run.str.endswith("trend")]
print(trend[["run", "block_months", "mean_difference_ann", "family_ci_low", "family_ci_high"]].to_string(index=False))
periods = pd.read_csv(OUT / "subperiods.csv")
print()
print(periods.query("strategy == 'trend' and cost_bps == 2")[["band", "period", "sharpe", "mean_difference_ann"]].to_string(index=False))
print()
print(pd.read_json(OUT / "candidate_screen.json").to_string(index=False))

                         run  block_months  mean_difference_ann  family_ci_low  family_ci_high
correlated_band00_1bps_trend             6               0.0031         0.0002          0.0061
correlated_band00_2bps_trend             6               0.0029         0.0000          0.0059
correlated_band05_1bps_trend             6               0.0015        -0.0000          0.0031
correlated_band05_2bps_trend             6               0.0014        -0.0001          0.0030
correlated_band00_1bps_trend            12               0.0031         0.0003          0.0062
correlated_band00_2bps_trend            12               0.0029         0.0001          0.0060
correlated_band05_1bps_trend            12               0.0015         0.0000          0.0031
correlated_band05_2bps_trend            12               0.0014        -0.0001          0.0031
correlated_band00_1bps_trend            24               0.0031         0.0004          0.0059
correlated_band00_2bps_trend            24        

## Interpretation

- Under DCC-GARCH the 10% band no longer pays for itself. TREND gains about 29 bps/yr at 2 bps cost with no band and 14 bps/yr with a 5% band. Almost all of it is gross return from tracking the target more closely; the extra fees are only 1-4 bps/yr.
- Only the no-band candidate meets every rule, and only just: its adjusted lower bounds are 0.00-0.03% a year, and the 5% band fails the interval rule.
- The gain sits in 1976-1994 (+0.5% to +0.6%/yr); 1995-2014 is flat. STATIC does not benefit (about +5 bps).
- Not adopted. The rule asks for drawdown, calibration and tracking review before any change, and this grid is in-sample. Turnover is 10.5x a year with no band versus 8.7x with 10%.